# 02A — Tracking environment diagnostic (no system changes)

Run this **instead of** the current tracking pilot for now. It runs small tests in child Python processes using the same interpreter as the current Jupyter kernel. It does not install/uninstall packages, modify PATH permanently, delete DLLs, or use `KMP_DUPLICATE_LIB_OK`.

Purpose: identify the exact import/inference stage that triggers the duplicate OpenMP runtime error. It tests with no manual DLL preloading (the previous worker explicitly preloaded PyTorch's DLL, which we should not assume is the correct fix).

Do not run Notebook 03 until the final YOLO one-frame inference test succeeds.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, textwrap, time
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\admin\ViTPose_Action_Recognition")
NOTEBOOK2_DIR = PROJECT_DIR / "Notebook_2_Improved_v2"
MANIFEST_PATH = PROJECT_DIR / "data" / "processed" / "manifest.csv"
WEIGHT_CANDIDATES = [PROJECT_DIR / "notebooks" / "yolo11n.pt", PROJECT_DIR / "yolo11n.pt"]
WEIGHTS = next((p for p in WEIGHT_CANDIDATES if p.is_file()), None)
LOG_DIR = NOTEBOOK2_DIR / "logs" / "openmp_diagnostic_v1"
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert MANIFEST_PATH.is_file(), f"Manifest not found: {MANIFEST_PATH}"
assert WEIGHTS is not None, "YOLO weights not found in expected project locations."
manifest = pd.read_csv(MANIFEST_PATH)
required = {"video_path", "label"}
missing = required - set(manifest.columns)
assert not missing, f"Manifest missing columns: {sorted(missing)}"
rows = manifest[manifest["label"].astype(str).str.casefold() == "typing".casefold()].sort_values("video_path")
assert not rows.empty, "No Typing video found in manifest."
VIDEO = Path(str(rows.iloc[0]["video_path"]))
assert VIDEO.is_file(), f"Video missing: {VIDEO}"

print("Interpreter:", sys.executable)
print("YOLO weights:", WEIGHTS)
print("Diagnostic video:", VIDEO)
print("Logs:", LOG_DIR)


Interpreter: C:\Users\admin\anaconda3\python.exe
YOLO weights: C:\Users\admin\ViTPose_Action_Recognition\notebooks\yolo11n.pt
Diagnostic video: C:\Users\admin\ViTPose_Action_Recognition\data\raw\ucf_sports\Typing\v_Typing_g01_c01.avi
Logs: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\openmp_diagnostic_v1


In [2]:
# Each test runs in a fresh child process. No parent Jupyter imports of torch/cv2 are changed.
# Import checks are deliberately separate so we can see which combination fails.

BASE = r'''import os, sys, traceback
for k in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[k] = "1"
os.environ.pop("KMP_DUPLICATE_LIB_OK", None)
print("CHILD_START", flush=True)
TEST = sys.argv[1]
try:
    if TEST == "numpy":
        import numpy as np; print("NUMPY_OK", np.__version__, flush=True)
    elif TEST == "cv2":
        import cv2; cv2.setNumThreads(1); print("CV2_OK", cv2.__version__, flush=True)
    elif TEST == "torch":
        import torch; torch.set_num_threads(1); print("TORCH_OK", torch.__version__, flush=True)
    elif TEST == "torch_cv2":
        import torch; torch.set_num_threads(1); print("TORCH_IMPORTED", flush=True)
        import cv2; cv2.setNumThreads(1); print("CV2_IMPORTED_AFTER_TORCH", cv2.__version__, flush=True)
    elif TEST == "cv2_torch":
        import cv2; cv2.setNumThreads(1); print("CV2_IMPORTED", flush=True)
        import torch; torch.set_num_threads(1); print("TORCH_IMPORTED_AFTER_CV2", torch.__version__, flush=True)
    elif TEST == "ultralytics":
        from ultralytics import YOLO; print("ULTRALYTICS_IMPORT_OK", flush=True)
    elif TEST == "yolo_load":
        from ultralytics import YOLO
        model = YOLO(sys.argv[2]); print("YOLO_LOAD_OK", flush=True)
    elif TEST == "one_frame_predict":
        import cv2
        from ultralytics import YOLO
        video = sys.argv[3]
        cap = cv2.VideoCapture(video)
        ok, frame = cap.read(); cap.release()
        if not ok: raise RuntimeError("Could not read first frame")
        print("FRAME_READ_OK", frame.shape, flush=True)
        model = YOLO(sys.argv[2]); print("YOLO_LOAD_OK", flush=True)
        result = model.predict(frame, classes=[0], conf=0.20, imgsz=320, device="cpu", max_det=5, verbose=False)[0]
        print("ONE_FRAME_PREDICT_OK", 0 if result.boxes is None else len(result.boxes), flush=True)
    else:
        raise ValueError(TEST)
except BaseException as e:
    print("PYTHON_EXCEPTION", repr(e), flush=True); traceback.print_exc(); sys.exit(2)
'''

TESTS = ["numpy", "cv2", "torch", "torch_cv2", "cv2_torch", "ultralytics", "yolo_load", "one_frame_predict"]
results = []
child_env = os.environ.copy()
for k in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    child_env[k] = "1"
child_env.pop("KMP_DUPLICATE_LIB_OK", None)

for name in TESTS:
    script = LOG_DIR / f"test_{name}.py"
    stdout_path = LOG_DIR / f"test_{name}_stdout.log"
    stderr_path = LOG_DIR / f"test_{name}_stderr.log"
    script.write_text(BASE, encoding="utf-8")
    args = [sys.executable, str(script), name]
    if name in ("yolo_load", "one_frame_predict"):
        args.append(str(WEIGHTS))
    if name == "one_frame_predict":
        args.append(str(VIDEO))
    try:
        p = subprocess.run(args, cwd=str(PROJECT_DIR), env=child_env, capture_output=True,
                           text=True, errors="replace", timeout=180)
        rc, stdout, stderr = p.returncode, p.stdout or "", p.stderr or ""
    except subprocess.TimeoutExpired as e:
        rc, stdout, stderr = 124, str(e.stdout or ""), str(e.stderr or "") + "\nTIMEOUT after 180 seconds"
    stdout_path.write_text(stdout, encoding="utf-8", errors="replace")
    stderr_path.write_text(stderr, encoding="utf-8", errors="replace")
    results.append({"test": name, "return_code": rc,
                    "stdout_tail": stdout[-1200:], "stderr_tail": stderr[-1800:],
                    "stdout_log": str(stdout_path), "stderr_log": str(stderr_path)})
    print(f"{name:18s} return_code={rc}")
    if rc != 0:
        print("  stdout:", stdout[-400:].replace("\n", " | "))
        print("  stderr:", stderr[-700:].replace("\n", " | "))

summary = pd.DataFrame([{k:v for k,v in r.items() if k not in ("stdout_tail", "stderr_tail")} for r in results])
summary_path = LOG_DIR / "diagnostic_summary.csv"
summary.to_csv(summary_path, index=False)
print("\nSaved diagnostic summary:", summary_path)


numpy              return_code=0
cv2                return_code=0
torch              return_code=0
torch_cv2          return_code=0
cv2_torch          return_code=0
ultralytics        return_code=0
yolo_load          return_code=0
one_frame_predict  return_code=0

Saved diagnostic summary: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\openmp_diagnostic_v1\diagnostic_summary.csv


In [3]:
# Compact final summary and first failing test
for r in results:
    print("\n==========", r["test"], "return_code", r["return_code"], "==========")
    if r["stdout_tail"].strip(): print("STDOUT:\n", r["stdout_tail"][-900:])
    if r["stderr_tail"].strip(): print("STDERR:\n", r["stderr_tail"][-1400:])

failed = [r["test"] for r in results if r["return_code"] != 0]
print("\nFailed tests:", failed if failed else "None")
print("Diagnostic folder:", LOG_DIR)



========== numpy return_code 0 ==========
STDOUT:
 CHILD_START
NUMPY_OK 2.1.3


========== cv2 return_code 0 ==========
STDOUT:
 CHILD_START
CV2_OK 4.13.0


========== torch return_code 0 ==========
STDOUT:
 CHILD_START
TORCH_OK 2.7.1+cu118


========== torch_cv2 return_code 0 ==========
STDOUT:
 CHILD_START
TORCH_IMPORTED
CV2_IMPORTED_AFTER_TORCH 4.13.0


========== cv2_torch return_code 0 ==========
STDOUT:
 CHILD_START
CV2_IMPORTED
TORCH_IMPORTED_AFTER_CV2 2.7.1+cu118


========== ultralytics return_code 0 ==========
STDOUT:
 CHILD_START
ULTRALYTICS_IMPORT_OK


========== yolo_load return_code 0 ==========
STDOUT:
 CHILD_START
YOLO_LOAD_OK


========== one_frame_predict return_code 0 ==========
STDOUT:
 CHILD_START
FRAME_READ_OK (240, 320, 3)
YOLO_LOAD_OK
ONE_FRAME_PREDICT_OK 1


Failed tests: None
Diagnostic folder: C:\Users\admin\ViTPose_Action_Recognition\Notebook_2_Improved_v2\logs\openmp_diagnostic_v1


## How to interpret this

- If `numpy`, `cv2`, and `torch` pass but `torch_cv2` or `cv2_torch` fails, the conflict is an import-time native-library interaction.
- If all imports pass but `yolo_load` fails, inspect Ultralytics/PyTorch model loading.
- If `yolo_load` passes but `one_frame_predict` fails, the OpenMP conflict appears during inference.
- This notebook makes no persistent package or environment changes. Keep Notebook 03 paused until `one_frame_predict` returns code 0.

Send the summary plus the `stderr` log for the first failed test. Do not uninstall packages or set `KMP_DUPLICATE_LIB_OK` based on this diagnostic alone.